# asyncio, demonstrated rather than described

Three fake LLM calls, run sequentially then concurrently — time both. Then break concurrency with a blocking call inside a coroutine, and fix it with `asyncio.to_thread`.

In [2]:
import asyncio
import time


async def fake_llm_call(name, delay=1):
    print(f"{name}: starting")
    await asyncio.sleep(delay)
    print(f"{name}: done")
    return f"{name} result"

## Sequential

Each `await` blocks until that call finishes before the next one starts. Three 1-second calls, one after another, should take about 3 seconds.

In [3]:
start = time.perf_counter()

r1 = await fake_llm_call("call-1")
r2 = await fake_llm_call("call-2")
r3 = await fake_llm_call("call-3")

print(f"sequential took {time.perf_counter() - start:.2f}s")

call-1: starting
call-1: done
call-2: starting
call-2: done
call-3: starting
call-3: done
sequential took 3.02s


## Concurrent with `asyncio.gather`

`gather` starts all three coroutines and lets them interleave on the event loop while each is `await`ing its `asyncio.sleep`. Same three 1-second calls, but now they overlap — total time should be close to 1 second, not 3.

In [4]:
start = time.perf_counter()

results = await asyncio.gather(
    fake_llm_call("call-1"),
    fake_llm_call("call-2"),
    fake_llm_call("call-3"),
)

print(f"concurrent took {time.perf_counter() - start:.2f}s")
print(results)

call-1: starting
call-2: starting
call-3: starting
call-1: done
call-2: done
call-3: done
concurrent took 1.00s
['call-1 result', 'call-2 result', 'call-3 result']


## Break it: a blocking call inside a coroutine

`time.sleep` is synchronous — it doesn't hand control back to the event loop the way `await asyncio.sleep` does. Put it inside an `async def` and the function is technically a coroutine, but nothing else can run while it sleeps. `gather` stops helping; three 1-second blocking calls take 3 seconds again, same as the sequential version.

In [5]:
async def blocking_llm_call(name, delay=1):
    print(f"{name}: starting")
    time.sleep(delay)  # blocks the whole event loop, not just this coroutine
    print(f"{name}: done")
    return f"{name} result"


start = time.perf_counter()

results = await asyncio.gather(
    blocking_llm_call("call-1"),
    blocking_llm_call("call-2"),
    blocking_llm_call("call-3"),
)

print(f"'concurrent' but blocking took {time.perf_counter() - start:.2f}s")

call-1: starting
call-1: done
call-2: starting
call-2: done
call-3: starting
call-3: done
'concurrent' but blocking took 3.00s


## Fix it with `asyncio.to_thread`

`asyncio.to_thread` runs the blocking call in a separate OS thread and awaits its result, so the event loop stays free to run the other two calls while it waits. Same blocking `time.sleep` underneath, but concurrency is back — close to 1 second, not 3.

This is the exact shape we'll use for the sync SQLAlchemy engine later: wrap the blocking DB call in `asyncio.to_thread` instead of switching to an async driver.

In [6]:
def blocking_sleep(name, delay=1):
    print(f"{name}: starting")
    time.sleep(delay)
    print(f"{name}: done")
    return f"{name} result"


start = time.perf_counter()

results = await asyncio.gather(
    asyncio.to_thread(blocking_sleep, "call-1"),
    asyncio.to_thread(blocking_sleep, "call-2"),
    asyncio.to_thread(blocking_sleep, "call-3"),
)

print(f"to_thread took {time.perf_counter() - start:.2f}s")

call-1: starting
call-2: starting
call-3: starting
call-2: done
call-3: done
call-1: done
to_thread took 1.01s
